# 基于 MindSpore 的 TorchDrug 论文简化复现

本 Notebook 复现论文 **TorchDrug: A Powerful and Flexible Machine Learning Platform for Drug Discovery** 中分子性质预测任务的核心思想。这里不直接调用 TorchDrug 或 PyTorch 训练代码，而是使用 RDKit 处理 SMILES，并用 MindSpore 实现 GIN/GAT、loss、训练循环和评估。

## 1. 检查运行环境

首先确认 Python、MindSpore、RDKit 等依赖是否已经安装。若这里导入失败，请先按照 README.md 创建环境。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

## 2. 导入 MindSpore 与项目模块

项目中的 `src.dataset` 负责数据下载、SMILES 解析、TorchDrug-like atom/bond 特征、图构造和 batch padding；`src.models` 负责 GIN/GAT；`src.trainer` 负责训练与评估。

In [ ]:
import mindspore as ms
import numpy as np

from src.dataset import load_dataset, split_dataset
from src.models import build_model
from src.trainer import TrainConfig, fit, evaluate

ms.set_context(mode=ms.PYNATIVE_MODE, device_target="CPU")
ms.set_seed(0)
np.random.seed(0)

## 3. 加载数据集并构造分子图

这里以 BACE 为例。数据从公开 CSV 下载，RDKit 将 SMILES 转为分子对象，再提取 TorchDrug default atom feature、bond feature 和邻接矩阵。HIV 只需要把 `dataset_name` 改为 `"hiv"`。

In [ ]:
dataset_name = "bace"
dataset = load_dataset(dataset_name, PROJECT_ROOT / "data")

print("Dataset:", dataset.name)
print("Num molecules:", len(dataset))
print("Node feature dim:", dataset.node_feature_dim)
print("Edge feature dim:", dataset.edge_feature_dim)
print("First SMILES:", dataset[0]["smiles"])
print("First graph nodes:", dataset[0]["node_feature"].shape[0])

## 4. 划分训练集、验证集和测试集

默认使用 scaffold split，比例为 8:1:1。scaffold split 更接近真实虚拟筛选场景，因为测试集分子骨架与训练集不同。

In [ ]:
train_set, valid_set, test_set = split_dataset(dataset, split="scaffold", seed=0)
print("Train / Valid / Test:", len(train_set), len(valid_set), len(test_set))

## 5. 构建 GIN 模型

GIN 通过邻居节点求和和 MLP 更新节点表示。TorchDrug-like 版本会使用 edge feature、BatchNorm、shortcut、concat hidden 和 sum readout。

In [ ]:
gin_model = build_model(
    "gin",
    input_dim=dataset.node_feature_dim,
    edge_input_dim=dataset.edge_feature_dim,
    hidden_dim=128,
    num_layer=4,
    num_head=4,
    dropout=0.1,
    variant="torchdrug_like",
)
print(gin_model)

## 6. 训练并评估 GIN

为了快速演示，下面默认只训练 3 个 epoch。正式实验建议使用 50 个 epoch 或更多，并至少运行 3 个随机种子。

In [ ]:
config = TrainConfig(epoch=3, batch_size=32, lr=1e-3, seed=0)
valid_metrics, test_metrics = fit(gin_model, train_set, valid_set, test_set, config)

print("Best valid:", valid_metrics)
print("Matched test:", test_metrics)

## 7. 构建并训练 GAT 模型

GAT 使用多头注意力为相邻节点分配不同权重。TorchDrug-like 版本会把 edge feature 加入 attention key，并只在分子图边和自环上计算注意力。

In [ ]:
gat_model = build_model(
    "gat",
    input_dim=dataset.node_feature_dim,
    edge_input_dim=dataset.edge_feature_dim,
    hidden_dim=128,
    num_layer=4,
    num_head=4,
    dropout=0.1,
    variant="torchdrug_like",
)

config = TrainConfig(epoch=3, batch_size=32, lr=1e-3, seed=0)
valid_metrics, test_metrics = fit(gat_model, train_set, valid_set, test_set, config)

print("Best valid:", valid_metrics)
print("Matched test:", test_metrics)

## 8. 命令行运行完整实验

Notebook 适合展示流程；正式实验建议使用命令行脚本，这样结果会自动追加到 `results/experiment_results.csv`。

In [ ]:
# 在 notebook 中也可以这样调用；正式运行时建议在终端执行。
# !python ../run_experiment.py --dataset all --model all --epoch 50 --batch_size 64 --seed 0 --device_target GPU --variant torchdrug_like

## 9. 结果记录

实验结果表包含 dataset、model、seed、valid/test AUROC 和 valid/test AUPRC。报告中可将多个 seed 的结果汇总为均值和标准差。